In [1]:
from pathlib import Path
import pandas as pd

raw_dir = Path(
    r"C:\Users\ADMIN\Downloads\beijing_air_quality\PRSA_Data_20130301-20170228"
)
csv_files = sorted(raw_dir.glob("PRSA_Data_*.csv"))

if not csv_files:
    raise FileNotFoundError(f"Aucun fichier PRSA_Data_*.csv dans {raw_dir}")

frames = []
for file in csv_files:
    frame = pd.read_csv(file)
    frame["station"] = (
        file.stem
        .replace("PRSA_Data_", "")
        .replace("_20130301-20170228", "")
    )
    frames.append(frame)

raw = pd.concat(frames, ignore_index=True)

raw["event_time"] = pd.to_datetime(
    raw[["year", "month", "day", "hour"]],
    errors="coerce",
)
raw["station"] = raw["station"].astype("string").str.strip()

clean = raw.dropna(subset=["event_time", "station"]).copy()
clean = clean[clean["station"] != ""]
clean = clean.drop_duplicates(subset=["station", "event_time"])
clean["event_date"] = clean["event_time"].dt.date
clean = clean.rename(columns={"PM2.5": "pm25"})

output_dir = Path(
    r"C:\Users\ADMIN\Desktop\AirWatch\analytics\outputs\prepared_data"
)
output_dir.mkdir(parents=True, exist_ok=True)
clean.to_parquet(output_dir / "observations_clean.parquet", index=False)

print(f"Lignes brutes : {len(raw):,}")
print(f"Lignes préparées : {len(clean):,}")
print(f"Fichier sauvegardé : {output_dir / 'observations_clean.parquet'}")

Lignes brutes : 420,768
Lignes préparées : 420,768
Fichier sauvegardé : C:\Users\ADMIN\Desktop\AirWatch\analytics\outputs\prepared_data\observations_clean.parquet


In [2]:
from pathlib import Path
import pandas as pd

raw_dir = Path(
    r"C:\Users\ADMIN\Downloads\beijing_air_quality\PRSA_Data_20130301-20170228"
)
csv_files = sorted(raw_dir.glob("PRSA_Data_*.csv"))

if not csv_files:
    raise FileNotFoundError(f"Aucun fichier PRSA_Data_*.csv dans {raw_dir}")

frames = []
for file in csv_files:
    frame = pd.read_csv(file)
    frame["station"] = (
        file.stem
        .replace("PRSA_Data_", "")
        .replace("_20130301-20170228", "")
    )
    frames.append(frame)

raw = pd.concat(frames, ignore_index=True)

numeric_columns = [
    "No", "year", "month", "day", "hour", "PM2.5", "PM10",
    "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM",
]
for column in numeric_columns:
    if column in raw.columns:
        raw[column] = pd.to_numeric(raw[column], errors="coerce")

raw["event_time"] = pd.to_datetime(
    raw[["year", "month", "day", "hour"]],
    errors="coerce",
)
raw["station"] = raw["station"].astype("string").str.strip()

clean = raw.dropna(subset=["event_time", "station"]).copy()
clean = clean[clean["station"] != ""]
clean = clean.drop_duplicates(subset=["station", "event_time"])
clean["event_date"] = clean["event_time"].dt.date
clean = clean.rename(columns={"PM2.5": "pm25"})

output_dir = Path(
    r"C:\Users\ADMIN\Desktop\AirWatch\analytics\outputs\prepared_data"
)
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "observations_clean.parquet"
clean.to_parquet(output_file, index=False)

print(f"Lignes brutes : {len(raw):,}")
print(f"Lignes préparées : {len(clean):,}")
print(f"Fichier sauvegardé : {output_file}")

Lignes brutes : 420,768
Lignes préparées : 420,768
Fichier sauvegardé : C:\Users\ADMIN\Desktop\AirWatch\analytics\outputs\prepared_data\observations_clean.parquet
